## Visualize Fontanka Results

Notebook based on 
* `/nfs/turbo/umms-minjilab/sionkim/miajet/notebooks/Bcell_vs_DP.ipynb`



In [1]:
import pandas as pd
import numpy as np
import os

import pybedtools

def precision_recall_f1(tp, fp, fn):
    if tp == 0 and fp == 0 and fn == 0:
        return 1.0, 1.0, 1.0
        
    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

def find_intersections(df_pred_in, df_true_in, leniency):
    """
    The df_pred must have the followign columns ['chrom', 'root', 'unique_id']

    The df_true is assumed to be the loading file given to the 3DPolyS-LE simulation,
    which has columns ['name', 'position', 'length', 'factor']. The only necessary column here is
    the 'position'
    """
    if df_pred_in.empty:
        # No predictions provided, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=0, fn=len(df_true_in) if df_true_in is not None else 0)
        result_dict = {
            "tp": 0,
            "fp": 0,
            "fn": len(df_true_in) if df_true_in is not None else 0,
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return pd.DataFrame(), result_dict

    if df_true_in is None:
        # No true data provided, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=len(df_pred_in), fn=0)
        result_dict = {
            "tp": 0,
            "fp": len(df_pred_in),
            "fn": 0,
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return pd.DataFrame(), result_dict

    df_pred = df_pred_in.copy()
    df_true = df_true_in.copy()

    df_pred["root"] = df_pred["root"].astype(int)

    # Need to map from genomic coordinates to bin coordiantes, which appears to 
    # differ by resolution and factor 2
    df_true["position"] /= 2 
    df_true["position"] = df_true["position"].astype(int)

    # Make BED dataframes for pybedtools intersect
    df_pred_bed = df_pred.assign(
        start=lambda d: np.clip(d["root"] - int(leniency), 0, None),
        end=lambda d: d["root"] + int(leniency) + 1 # half-open BED interval
    )[["chrom", "start", "end", "unique_id", "root"]]


    chrom = df_pred["chrom"].iloc[0]
    df_true_bed = df_true.assign(
        chrom=chrom,
        start=lambda d: np.clip(d["position"] - int(leniency), 0, None),
        end=lambda d: d["position"] + int(leniency) + 1,
    )[["chrom", "start", "end", "name", "position"]]

    pred_bed = pybedtools.BedTool.from_dataframe(df_pred_bed)
    true_bed = pybedtools.BedTool.from_dataframe(df_true_bed)

    hits = pred_bed.intersect(true_bed, wa=True, wb=True)

    intersections = hits.to_dataframe(names=[
        "chrom", "pred_start", "pred_end", "unique_id", "pred_root",
        "true_chrom", "true_start", "true_end", "true_name", "true_position"
    ])

    if intersections.empty:
        # No matches found, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=len(df_pred), fn=len(df_true))
        result_dict = {
            "tp": 0,
            "fp": len(df_pred),
            "fn": len(df_true),
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return intersections, result_dict

    # --- Deduplicate: one-to-one matching by closest distance ---
    intersections["distance"] = (intersections["pred_root"] - intersections["true_position"]).abs()

    # For each true jet, keep only the closest predicted match
    intersections = intersections.sort_values("distance")
    intersections = intersections.drop_duplicates(subset="true_name", keep="first")

    # Also ensure each prediction only maps to one true jet (the closest)
    intersections = intersections.drop_duplicates(subset="unique_id", keep="first")

    # Compute scores
    matched_pred_ids = set(intersections["unique_id"].unique()) # TP
    all_pred_ids = set(df_pred["unique_id"].unique())
    matched_true_names = set(intersections["true_name"].unique()) 
    all_true_names = set(df_true["name"].unique())

    tp = len(matched_pred_ids)
    fp = len(all_pred_ids - matched_pred_ids)
    fn = len(all_true_names - matched_true_names)

    p, r, f = precision_recall_f1(tp=tp, fp=fp, fn=fn)

    result_dict = {
        "tp": tp,
        "fp" : fp,
        "fn" : fn,
        "precision": p,
        "recall": r,
        "f1": f,
    }

    return intersections, result_dict
    

## Performance

In [2]:
leniency = 25e3 * 2 # ± Uncertainty for matching predicted fountain to true je

In [3]:
frames = []

for resolution in [5000, 10000, 25000, 50000]:

    # Previously, we optimized
    # if resolution == 5000:
    #     window_sizes = [
    #         int(150000),
    #         int(150000),
    #         int(150000),
    #         int(150000),
    #         int(150000),
    #         int(150000),
    #         int(150000),
    #         int(150000),
    #         int(150000),
    #     ]
    # elif resolution == 10000:
    #     window_sizes = [
    #         int(300e3), # Either 300e3 or 1e6 or 5e6
    #         int(300e3),
    #         int(300e3),
    #         int(300e3),
    #         int(300e3), # Either 100000 or 300000 or 500000 
    #         int(300e3),
    #         int(300e3),
    #         int(300e3),
    #         int(300e3),
    #     ]
    # elif resolution == 25000:
    #     window_sizes = [
    #         int(500000),
    #         int(500000),
    #         int(500000),
    #         int(500000),
    #         int(500000),
    #         int(500000),
    #         int(500000),
    #         int(500000),
    #         int(500000),
    #     ]
    # elif resolution == 50000:
    #     window_sizes = [
    #         int(1e6), # Either 1e6 or 5e6
    #         int(1e6),
    #         int(1e6),
    #         int(1e6),
    #         int(500000), # 500000
    #         int(1e6),
    #         int(1e6),
    #         int(1e6),
    #         int(1e6),
    #     ]

    window_sizes = [
        int(500000),
        int(500000),
        int(500000),
        int(500000),
        int(500000),
        int(500000),
        int(500000),
        int(500000),
        int(500000),
    ]

    hic_files=[
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1b_s-23/out_fast_bounded_jets/r2.14/test1b_s-23_hic_003.hic",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/out_fast_unbounded_jets/r2.14/test1a_s-23-fast_hic_003.hic",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/out_fast_unbounded_jets/r2.14/chromsim1-fast-unbounded_hic_003.hic",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/out_fast_bounded_jets/r2.14/chromsim1-fast-bounded_hic_003.hic",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/out_slow_unbounded_jets/r2.14/test1a_s-23-slow_hic_003.hic", # Should be much smaller window size and highest resolution possible
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1c_s-23/out_fast_bounded_jets/r2.14/test1c_s-23_hic_003.hic",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1cii_s-23/out_fast_bounded_jets/r2.14/test1cii_s-23_hic_003.hic",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1e_s-23/out_tads/r2.14/test1e_s-23_hic_003.hic",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1eii_s-23/out_tads/r2.14/test1eii_s-23_hic_003.hic",
    ]

    # SPECIFY THE **NON-THRESHOLDED** FILES
    # We will generate results for both
    fontanka_files = [
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[0]}/FONTANKA_test1b_s-23_hic_003.{resolution}.predicted.fountains.tsv",
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[1]}/FONTANKA_test1a_s-23-fast_hic_003.{resolution}.predicted.fountains.tsv",
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[2]}/FONTANKA_chromsim1-fast-unbounded_hic_003.{resolution}.predicted.fountains.tsv",
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[3]}/FONTANKA_chromsim1-fast-bounded_hic_003.{resolution}.predicted.fountains.tsv",
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[4]}/FONTANKA_test1a_s-23-slow_hic_003.{resolution}.predicted.fountains.tsv",
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[5]}/FONTANKA_test1c_s-23_hic_003.{resolution}.predicted.fountains.tsv",
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[6]}/FONTANKA_test1cii_s-23_hic_003.{resolution}.predicted.fountains.tsv",
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[7]}/FONTANKA_test1e_s-23_hic_003.{resolution}.predicted.fountains.tsv",
        f"/nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_{resolution}_{window_sizes[8]}/FONTANKA_test1eii_s-23_hic_003.{resolution}.predicted.fountains.tsv",
    ]

    jet_loc_files = [
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1b_s-23/loading.csv",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/narrow_loading_50.csv",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/JET-fast-unbounded_loading_1_100.csv",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/JET-fast-bounded_loading_1_100.csv",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/diffuse_loading_10000.csv",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1c_s-23/loading.csv",
        "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1cii_s-23/loading.csv",
        None, # no true jets (TADs)
        None, # no true jets (TADs)
    ]

    # Instantiate 3 independent result dicts for the 3 different thresholding modes
    result_dict_pool_unthresholded = {
        "tp": 0,
        "fp": 0,
        "fn": 0,
        "n_true": 0,
        "n_reported": 0,
        "resolution": resolution,
    }
    result_dict_pool_thresholded = result_dict_pool_unthresholded.copy()
    result_dict_pool_thresholded_weak = result_dict_pool_unthresholded.copy()


    # Fontanka
    for i in range(len(hic_files)):

        f_hic_file = hic_files[i]
        f_pred_fontanka = fontanka_files[i]

        # NEW: load true
        f_true_jet_loc = jet_loc_files[i]
        if f_true_jet_loc is not None:
            df_true = pd.read_csv(f_true_jet_loc)
        else:
            df_true = None

        
        if not os.path.exists(f_hic_file):
            print(f"Hi-C file not found: {f_hic_file}")
            break
        if not os.path.exists(f_pred_fontanka):
            print(f"FONTANKA prediction file not found: {f_pred_fontanka}")
            break
        
        f_pred_fontanka_thresholded = f_pred_fontanka.removesuffix('.tsv') + '.thresholded.tsv'

        if not os.path.exists(f_pred_fontanka_thresholded):
            print(f"FONTANKA thresholded file not found: {f_pred_fontanka_thresholded}")
            break

        f_pred_fontanka_thresholded_weak = f_pred_fontanka.removesuffix('.tsv') + '.thresholded.weak.tsv'

        if not os.path.exists(f_pred_fontanka_thresholded_weak):
            print(f"FONTANKA weak thresholded file not found: {f_pred_fontanka_thresholded_weak}")
            break

        save_dir = os.path.dirname(f_pred_fontanka)

        chrom = "chrS"

        result_set = [f_pred_fontanka, f_pred_fontanka_thresholded, f_pred_fontanka_thresholded_weak]
        result_thresh_mode = ["unthresholded", "thresholded", "thresholded_weak"]

        for j, f in enumerate(result_set):

            # Load it in
            fontanka_table = pd.read_csv(f, sep="\t", index_col=0)
            fontanka_table.dropna(inplace=True, how="any")
            fontanka_table.reset_index(inplace=True, drop=True)

            # Process FONTANKA
            fontanka_table = fontanka_table.reset_index(drop=True)
            fontanka_table["root"] = (fontanka_table["end"] + fontanka_table["start"]) / 2
            fontanka_table["unique_id"] = fontanka_table.index
            fontanka_table["extrusion_x"] = fontanka_table["window_end"]
            fontanka_table["extrusion_y"] = fontanka_table["window_start"]
            assert np.all(fontanka_table["extrusion_x"] >= fontanka_table["extrusion_y"])

            fontanka_table["length"] = -1

            fontanka_minimal = fontanka_table.copy()
            fontanka_minimal = fontanka_minimal[["unique_id", "chrom", "root", "extrusion_x", "extrusion_y"]]


            # NEW
            _, result_dict = find_intersections(df_pred_in=fontanka_minimal, df_true_in=df_true, leniency=leniency)

            exp_name = os.path.basename(f_hic_file).strip(".hic")

            if j == 0:
                pool = result_dict_pool_unthresholded
            elif j == 1:
                pool = result_dict_pool_thresholded
            elif j == 2:
                pool = result_dict_pool_thresholded_weak

            for key in ["tp", "fp", "fn"]:
                pool[key] += result_dict[key]

            pool["n_true"]     += len(df_true) if df_true is not None else 0
            pool["n_reported"] += len(fontanka_minimal)

    
    result_dict_pool_unthresholded["precision"], result_dict_pool_unthresholded["recall"], result_dict_pool_unthresholded["f1"] = precision_recall_f1(
        tp=result_dict_pool_unthresholded["tp"],
        fp=result_dict_pool_unthresholded["fp"],
        fn=result_dict_pool_unthresholded["fn"],
    )

    result_dict_pool_thresholded["precision"], result_dict_pool_thresholded["recall"], result_dict_pool_thresholded["f1"] = precision_recall_f1(
        tp=result_dict_pool_thresholded["tp"],
        fp=result_dict_pool_thresholded["fp"],
        fn=result_dict_pool_thresholded["fn"],
    )

    result_dict_pool_thresholded_weak["precision"], result_dict_pool_thresholded_weak["recall"], result_dict_pool_thresholded_weak["f1"] = precision_recall_f1(
        tp=result_dict_pool_thresholded_weak["tp"],
        fp=result_dict_pool_thresholded_weak["fp"],
        fn=result_dict_pool_thresholded_weak["fn"],
    )

    result_dict_pool_unthresholded['thresh'] = 'unthresholded'
    result_dict_pool_thresholded['thresh'] = 'thresholded'
    result_dict_pool_thresholded_weak['thresh'] = 'thresholded_weak'

    for d in [result_dict_pool_unthresholded, result_dict_pool_thresholded, result_dict_pool_thresholded_weak]:
        assert d["n_true"] == d["tp"] + d["fn"], \
            f"[res={resolution}, {d['thresh']}] n_true={d['n_true']} != tp+fn={d['tp'] + d['fn']}"
        assert d["n_reported"] == d["tp"] + d["fp"], \
            f"[res={resolution}, {d['thresh']}] n_reported={d['n_reported']} != tp+fp={d['tp'] + d['fp']}"

    frames.append(result_dict_pool_unthresholded.copy())
    frames.append(result_dict_pool_thresholded.copy())
    frames.append(result_dict_pool_thresholded_weak.copy())


FONTANKA prediction file not found: /nfs/turbo/umms-minjilab/sionkim/fontanka/simulation_results_5000_500000/FONTANKA_test1b_s-23_hic_003.5000.predicted.fountains.tsv


In [ ]:
result_df = pd.DataFrame(frames)
result_df 

# Use best F1 score for standardized resolution 25kb as the reported version

,tp,fp,fn,n_true,n_reported,resolution,precision,recall,f1,thresh
0,0,0,0,0,0,5000,1.000000,1.000000,1.000000,unthresholded
1,0,0,0,0,0,5000,1.000000,1.000000,1.000000,thresholded
2,0,0,0,0,0,5000,1.000000,1.000000,1.000000,thresholded_weak
3,73,1172,18,91,1245,10000,0.058635,0.802198,0.109281,unthresholded
4,7,2,84,91,9,10000,0.777778,0.076923,0.140000,thresholded
5,62,81,29,91,143,10000,0.433566,0.681319,0.529915,thresholded_weak
6,71,538,20,91,609,25000,0.116585,0.780220,0.202857,unthresholded
7,5,3,86,91,8,25000,0.625000,0.054945,0.101010,thresholded
8,62,61,29,91,123,25000,0.504065,0.681319,0.579439,thresholded_weak
9,68,330,23,91,398,50000,0.170854,0.747253,0.278119,unthresholded
